# C01 · Code walkthrough: `core/` (ports, topology, margins)

**Track:** code walkthroughs. These notebooks go through `src/aircraft_closure` file by file. Every listing is printed
from the repository with its real line numbers (`show_source`), so it cannot drift from the code. Each module ends
with gotchas and review questions.

| Module | Lines | What it holds |
|---|---|---|
| `core/ports.py` | 57 | port domains, directions, port specs, port-value dataclasses, which field is a "flow" |
| `core/topology.py` | 215 | `Topology` (instances, buses, connections, validation) and `connection_residuals` |
| `core/margins.py` | 44 | normalized margins and the sorted margin report |

**Who imports `core`:** `powertrain/ports.py`, `powertrain/topologies.py`, `powertrain/compatibility.py`,
`performance/flight_point.py`, `mission/mission.py`, `thermal/heat.py`, and the examples (`halo_sizing.py`,
`series_hybrid_point.py`, ...). `core` itself imports nothing from the package: it is the bottom layer.

In [1]:
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(repo_root / "tutorials"))
from tutorial_helpers import *          # paths, plot style, check(), lb(), baseline(), capture_opti()

---
## 1. `core/ports.py`

In [2]:
show_source("src/aircraft_closure/core/ports.py")

# src/aircraft_closure/core/ports.py  lines 1-57
   1  """Typed physical ports; flow is positive in the port's nominal direction."""
   2  from dataclasses import dataclass
   3  from enum import Enum
   4  from typing import Any
   5  
   6  
   7  class Domain(Enum):
   8      MECHANICAL = "mechanical"
   9      ELECTRICAL = "electrical"
  10      FUEL = "fuel"
  11  
  12  
  13  class Direction(Enum):
  14      """Nominal power or fuel flow direction relative to the owning component."""
  15      IN = "in"
  16      OUT = "out"
  17  
  18  
  19  @dataclass(frozen=True)
  20  class PortSpec:
  21      name: str
  22      domain: Domain
  23      direction: Direction
  24  
  25  
  26  @dataclass(frozen=True)
  27  class MechanicalPortValue:
  28      """Shaft state; torque is positive when power flows in the nominal direction."""
  29      speed_rad_s: Any
  30      torque_Nm: Any
  31  
  32  
  33  @dataclass(frozen=True)
  34  class ElectricalPortValue:
  35      """Terminal s

**Line by line**

- **L7-10 `Domain`**: three physical domains. Mechanical (shaft), electrical (DC or AC terminals), fuel. There is no
  thermal domain: heat is handled as named loads (`thermal/heat.py`), not through ports.
- **L13-16 `Direction`**: the *nominal* direction of power (or fuel) flow relative to the component that owns the
  port. A motor's electrical port is IN, its shaft OUT; a generator is the reverse. Direction is a *sign convention*,
  not a restriction: a negative current on an OUT port means power flows backwards (the battery charging).
- **L19-23 `PortSpec`**: name + domain + direction. Ports are declared per component *type* in
  `powertrain/ports.py`, not inside the component classes (C07 covers why).
- **L26-42 port values**: the state on a port. `MechanicalPortValue(speed_rad_s, torque_Nm)`,
  `ElectricalPortValue(voltage_V, current_A)`, `FuelPortValue(fuel_flow_kg_s)`. Power is *derived* (ω·Q, V·I), never
  stored. Fields are typed `Any` on purpose: they may hold floats or CasADi expressions.
- **L45-49 `port_value_types`**: domain → value class. `connection_residuals` uses it to reject, for example, a
  mechanical value supplied for an electrical port (`TypeError`).
- **L51-57 `port_flow_fields`** (added in Tier 18 for redundancy): which field is the **flow** of each domain
  (torque, current, fuel flow). Flows add at a junction; the other fields are **efforts** (speed, voltage) and are
  equal at a junction. This is the classic effort/flow split of bond graphs, and it is exactly what a combiner or
  splitter needs.

**Run it**

In [3]:
from aircraft_closure.core.ports import (Direction, Domain, ElectricalPortValue, MechanicalPortValue, PortSpec,
                                         port_flow_fields, port_value_types)
shaft_out = PortSpec("shaft", Domain.MECHANICAL, Direction.OUT)
print(shaft_out)
value = MechanicalPortValue(speed_rad_s=200.0, torque_Nm=1500.0)
print(value, "-> power", value.speed_rad_s * value.torque_Nm / 1e3, "kW (derived, not stored)")
print({d.value: t.__name__ for d, t in port_value_types.items()})
print({d.value: f for d, f in port_flow_fields.items()})
# Port values may be symbolic:
opti = asb.Opti(); w = opti.variable(init_guess=100.0)
print(MechanicalPortValue(w, 10 * w))

PortSpec(name='shaft', domain=<Domain.MECHANICAL: 'mechanical'>, direction=<Direction.OUT: 'out'>)
MechanicalPortValue(speed_rad_s=200.0, torque_Nm=1500.0) -> power 300.0 kW (derived, not stored)
{'mechanical': 'MechanicalPortValue', 'electrical': 'ElectricalPortValue', 'fuel': 'FuelPortValue'}
{'mechanical': 'torque_Nm', 'electrical': 'current_A', 'fuel': 'fuel_flow_kg_s'}
MechanicalPortValue(speed_rad_s=MX((100*opti0_x_1)), torque_Nm=MX((1000*opti0_x_1)))


---
## 2. `core/topology.py`

The module docstring is the specification; read it first.

In [4]:
show_source("src/aircraft_closure/core/topology.py", 1, 63)

# src/aircraft_closure/core/topology.py  lines 1-63
   1  """Network description only: no variables, constraints, ordering or solves.
   2  
   3  A direct connection joins one OUT port to one IN port of the same domain and
   4  instance count. A bus is a junction for any number of electrical ports: all
   5  attached voltages are equal and currents obey Kirchhoff's current law. Cycles
   6  are allowed; `connection_residuals` returns expressions the caller constrains
   7  to zero in its own `asb.Opti`.
   8  
   9  An instance with `count` n represents n identical copies sharing one set of
  10  port values (symmetric operation). Its flow into a bus is scaled by n.
  11  
  12  Tier 18: a direct connection made with `combine=True` may join counts where
  13  one is an integer multiple of the other: a combiner (several lane motors on
  14  one gearbox input) or a splitter (one pack into isolated strings). Efforts
  15  (speed, voltage) are equal and the total flow is conserved,
  16 

**L1-24 docstring, the contract in four sentences:**
1. a direct connection joins one OUT port to one IN port of the same domain and the same count;
2. a bus joins any number of electrical ports: equal voltages, Kirchhoff's current law;
3. `count = n` means n identical copies sharing one set of port values, and their flow into a bus is scaled by n;
4. Tier 18: `combine=True` allows count mismatches that are integer multiples (combiner or splitter); a bus may have a
   count (identical cross-strapped buses lumped as one junction).

Cycles are allowed (L5-7): the topology never orders or solves anything, so a cycle is just more equations.

**L27-38 `Instance`**: name, component object, its ports, count. `get_port` is a linear search (a handful of ports).
**L41-45 `Bus`**: name, domain, count. **L48-55 `Connection`**: `source` is always the OUT side for direct
connections (normalized in `connect`, L136-137); for bus connections `source` is the port and `target` the bus name.
**L58-62 `Residual`**: a label and an expression the *caller* drives to zero. Units are in the label suffix.

In [5]:
show_source("src/aircraft_closure/core/topology.py", 65, 104)

# src/aircraft_closure/core/topology.py  lines 65-104
  65  class Topology:
  66      def __init__(self):
  67          self._instances = {}
  68          self._buses = {}
  69          self._connections = []
  70          self._connected_ports = set()
  71  
  72      @property
  73      def instances(self):
  74          return dict(self._instances)
  75  
  76      @property
  77      def buses(self):
  78          return dict(self._buses)
  79  
  80      @property
  81      def connections(self):
  82          return tuple(self._connections)
  83  
  84      def add(self, name, component, ports, count=1):
  85          self._check_new_name(name)
  86          if "." in name:
  87              raise ValueError(f"Instance name '{name}' must not contain '.'.")
  88          if not isinstance(count, int) or isinstance(count, bool) or count < 1:
  89              raise ValueError(f"Instance count must be a positive integer, got {count!r}.")
  90          port_names = [port.name for por

**L65-82**: state is kept in private containers; the properties return **copies** (`dict(...)`, `tuple(...)`), so a
caller cannot mutate the topology behind its back. Only `add`, `add_bus` and `connect` change it.

**L84-94 `add`** validates on entry:
- unique name across instances *and* buses (L85 → L155-157);
- no `"."` in the name (L86-87), because references are `"instance.port"` strings parsed with `partition(".")`;
- `count` must be a real positive `int` (L88-89). `isinstance(count, bool)` is excluded explicitly because `True` is an
  `int` in Python. **Counts are never symbolic**: structure is fixed before the optimizer sees anything;
- no duplicate port names (L90-92).

**L96-103 `add_bus`**: only electrical buses. Mechanical junctions must be expressed as combiners (L98-99): a shaft
junction is a gearbox input with several lanes, which the code models as `connect(..., combine=True)`.

In [6]:
show_source("src/aircraft_closure/core/topology.py", 105, 157)

# src/aircraft_closure/core/topology.py  lines 105-157
 105      def connect(self, reference_a, reference_b, combine=False):
 106          """Join two ports (either order) or a port and a bus; validated immediately.
 107  
 108          `combine=True` (Tier 18) allows a direct connection between counts where one is an integer multiple of
 109          the other (a combiner or splitter); without it the counts must be equal.
 110          """
 111          if reference_b in self._buses or reference_a in self._buses:
 112              port_reference, bus_name = ((reference_a, reference_b) if reference_b in self._buses
 113                                          else (reference_b, reference_a))
 114              instance, port = self._resolve(port_reference)
 115              bus = self._buses[bus_name]
 116              if port.domain is not bus.domain:
 117                  raise ValueError(f"Cannot connect {port.domain.value} port '{port_reference}' "
 118                             

**L105-143 `connect`**, the only place wiring rules live:

- **L111-119 bus branch:** either argument may be the bus. The port must exist (L114, `_resolve`) and be in the bus's
  domain (L116-118). A bus connection records `(port, bus)`.
- **L120-137 direct branch:**
  - same domain (L123-125);
  - exactly one OUT and one IN (L126-128: the set of the two directions must equal {OUT, IN});
  - counts equal, unless `combine` (L129-132);
  - with `combine`, the larger count must be a multiple of the smaller (L133-135): 4 lane motors onto 2 gearboxes
    is fine, 3 onto 2 is not;
  - **L136-137** normalize the connection so `source` is the OUT port, whatever order the caller passed.
- **L138-141 one connection per port.** Every non-bus reference is recorded in `_connected_ports`; connecting a port
  twice raises. Bus names are *not* added (L141 filter), so a bus takes any number of connections. This is what forces
  junctions to be explicit buses.
- **L148-153 `_resolve`** parses `"instance.port"`, raising `KeyError` for unknown instances, and `Instance.get_port`
  raises for unknown ports.

In [7]:
from aircraft_closure.core.topology import Topology

class Dummy:              # components can be any object: the topology never calls them
    pass

def fresh():
    t = Topology()
    t.add("motor", Dummy(), (PortSpec("electrical", Domain.ELECTRICAL, Direction.IN),
                             PortSpec("shaft", Domain.MECHANICAL, Direction.OUT)), count=4)
    t.add("gearbox", Dummy(), (PortSpec("shaft_in", Domain.MECHANICAL, Direction.IN),
                               PortSpec("shaft_out", Domain.MECHANICAL, Direction.OUT)), count=2)
    t.add("rotor", Dummy(), (PortSpec("shaft", Domain.MECHANICAL, Direction.IN),), count=2)
    t.add("battery", Dummy(), (PortSpec("electrical", Domain.ELECTRICAL, Direction.OUT),))
    t.add_bus("bus")
    return t

cases = {
    "count=True (a bool)": lambda t: t.add("x", Dummy(), (), count=True),
    "name with a dot": lambda t: t.add("a.b", Dummy(), ()),
    "mechanical bus": lambda t: t.add_bus("shaft_bus", Domain.MECHANICAL),
    "shaft to electrical bus": lambda t: t.connect("motor.shaft", "bus"),
    "shaft to electrical port": lambda t: t.connect("motor.shaft", "battery.electrical"),
    "IN to IN": lambda t: t.connect("gearbox.shaft_in", "rotor.shaft"),
    "4 -> 2 without combine": lambda t: t.connect("motor.shaft", "gearbox.shaft_in"),
    "same port twice": lambda t: (t.connect("motor.electrical", "bus"), t.connect("motor.electrical", "bus")),
    "unknown port": lambda t: t.connect("motor.torque", "gearbox.shaft_in"),
}
for label, action in cases.items():
    try:
        action(fresh())
        print(f"accepted   {label}")
    except (ValueError, KeyError) as error:
        print(f"rejected   {label:<26} {type(error).__name__}: {error}")

t = fresh()
c = t.connect("gearbox.shaft_in", "motor.shaft", combine=True)      # IN first: connect normalizes the order
print("\nnormalized:", c)

rejected   count=True (a bool)        ValueError: Instance count must be a positive integer, got True.
rejected   name with a dot            ValueError: Instance name 'a.b' must not contain '.'.
rejected   mechanical bus             ValueError: Only electrical buses are supported; shaft combiners use connect(..., combine=True).
rejected   shaft to electrical bus    ValueError: Cannot connect mechanical port 'motor.shaft' to electrical bus 'bus'.
rejected   shaft to electrical port   ValueError: Domain mismatch: 'motor.shaft' is mechanical, 'battery.electrical' is electrical.
rejected   IN to IN                   ValueError: Direct connections join one OUT and one IN port: 'gearbox.shaft_in' and 'rotor.shaft' are both in.
rejected   4 -> 2 without combine     ValueError: Direct connection between counts 4 and 2 needs combine=True (a combiner or splitter) or a bus.
rejected   same port twice            ValueError: Port 'motor.electrical' is already connected; use a bus for junctions.
rej

**`connection_residuals`: the equations a topology implies**

In [8]:
show_source("src/aircraft_closure/core/topology.py", 160, 215)

# src/aircraft_closure/core/topology.py  lines 160-215
 160  def connection_residuals(topology, port_values):
 161      """Return labelled residuals for every connection; the caller applies them.
 162  
 163      `port_values` maps "instance.port" to the matching port-value dataclass for
 164      every connected port. Values may be numeric or CasADi expressions.
 165      """
 166      residuals = []
 167      bus_ports = {name: [] for name in topology.buses}
 168      for connection in topology.connections:
 169          if connection.target in bus_ports:
 170              bus_ports[connection.target].append(connection.source)
 171              continue
 172          source = _port_value(topology, port_values, connection.source)
 173          target = _port_value(topology, port_values, connection.target)
 174          link = f"{connection.source}->{connection.target}"
 175          count_source = count_target = 1
 176          flow_field = None
 177          if connection.combine:
 1

- **L166-171:** bus connections are collected per bus (handled after the loop); everything else is a direct link.
- **L172-173:** `_port_value` (L208-215) fetches the caller's value for each side and type-checks it against the
  port's domain.
- **L175-189 direct links:** one residual per field of the port-value dataclass (`dataclasses.fields`), so a shaft
  link gives a speed residual and a torque residual. For a **combiner/splitter** (L177-181) the flow field gets
  `count_source × flow_source − count_target × flow_target` and is labelled "(total)"; efforts stay plain differences.
- **L191-204 buses:** voltages of all attached ports equal the first one's (L195-197); one Kirchhoff residual per bus
  (L199-204): OUT ports add `+count × current`, IN ports `−count × current`. A battery is an OUT port, so a positive
  battery current (discharge) feeds the bus.
- **What `Bus.count` does here: nothing.** Two cross-strapped buses are lumped as one junction under symmetric
  operation; the count is read only by the redundancy logic (`powertrain/redundancy.py`, C07).

Verify the combiner and the bus equations on numbers:

In [9]:
from aircraft_closure.core.topology import connection_residuals
t = fresh()
t.connect("motor.shaft", "gearbox.shaft_in", combine=True)
t.connect("battery.electrical", "bus")
t.connect("motor.electrical", "bus")
values = {
    "motor.shaft": MechanicalPortValue(150.0, 1000.0),            # each of 4 lane motors
    "gearbox.shaft_in": MechanicalPortValue(150.0, 2000.0),       # each of 2 gearboxes: 4 x 1000 = 2 x 2000
    "battery.electrical": ElectricalPortValue(700.0, 400.0),
    "motor.electrical": ElectricalPortValue(700.0, 100.0),        # 4 motors x 100 A = 400 A from the battery
}
for r in connection_residuals(t, values):
    print(f"{r.label:<45} {r.value}")
check("combiner conserves total torque and the bus conserves current",
      all(abs(r.value) < 1e-12 for r in connection_residuals(t, values)))
try:
    connection_residuals(t, {**values, "motor.shaft": ElectricalPortValue(1, 1)})
except TypeError as error:
    print("type check:", error)

motor.shaft->gearbox.shaft_in speed_rad_s     0.0
motor.shaft->gearbox.shaft_in torque_Nm (total) 0.0
bus motor.electrical voltage_V                0.0
bus current_A                                 0.0
PASS  combiner conserves total torque and the bus conserves current
type check: Port 'motor.shaft' needs MechanicalPortValue, got ElectricalPortValue.


**How the Halo uses this (see tutorial 02 for the demonstration):** the production flight point
(`performance/flight_point.py`, C11) does **not** call `connection_residuals`. It writes the connections by
construction and closes the bus with power balances; it builds `port_values` only for the operating margins. The
residuals are applied in `examples/series_hybrid_point.py` and in `tests/core/test_topology.py`. Because the residuals
use *installed* counts, they would be wrong in a failure state (one lane active), which is why degraded states live in
the flight condition, not in the topology.

---
## 3. `core/margins.py`

In [10]:
show_source("src/aircraft_closure/core/margins.py")

# src/aircraft_closure/core/margins.py  lines 1-44
   1  """Normalized margins: >= 0 is compatible, 0.1 means 10 % headroom.
   2  
   3  Margins report; they never clip, resize or solve. Callers either constrain
   4  `margin.value >= 0` in their own `asb.Opti` or report margins after a solve.
   5  """
   6  from dataclasses import dataclass
   7  from typing import Any
   8  
   9  
  10  @dataclass(frozen=True)
  11  class Margin:
  12      label: str
  13      value: Any
  14  
  15  
  16  @dataclass(frozen=True)
  17  class MarginReportEntry:
  18      label: str
  19      value: Any
  20      compatible: bool
  21  
  22  
  23  def margin_below(label, value, limit):
  24      """Margin for `value <= limit`, normalized by the (positive) limit."""
  25      return Margin(label, (limit - value) / limit)
  26  
  27  
  28  def margin_above(label, value, limit):
  29      """Margin for `value >= limit`, normalized by the (positive) limit."""
  30      return Margin(label, (value -

- **L1-5 docstring:** margins **report**. They never clip, resize or solve. That one sentence is why every limit in
  the sizing becomes a constraint `margin.value >= 0` written by the caller (`halo_sizing.py` constrains the whole
  list in one line) instead of a hidden `min()` inside a model.
- **L10-13 `Margin(label, value)`**; **L16-20 `MarginReportEntry`** adds the `compatible` flag after a solve.
- **L23-30:** `margin_below = (limit − value)/limit` for "value ≤ limit"; `margin_above = (value − limit)/limit` for
  "value ≥ limit". Both are normalized by the *limit*, which must be positive (stated in the docstrings, not checked:
  a negative limit would flip the sign of the inequality).
- **L33-44 `margin_report`:** maps each value through `value_of` (typically `solution.value`), sorts ascending
  (most critical first). The comment at L36-38 matters: comparing a symbolic value without `value_of` raises instead
  of being silently coerced to a number. The sizing reports a constraint as **binding** when |margin| < 1e-4
  (`halo_sizing.py`, in the result's `binding` field).

In [11]:
from aircraft_closure.core.margins import margin_above, margin_below, margin_report
margins = [margin_below("motor torque_Nm", 2400.0, 3000.0),        # 20 % headroom
           margin_above("static_margin", 0.10, 0.10),               # exactly on the limit: binding
           margin_below("battery current_A", 1200.0, 1125.0)]       # violated
for entry in margin_report(margins):
    print(f"{entry.value:+.4f}  compatible={entry.compatible}  {entry.label}")

opti = asb.Opti(); x = opti.variable(init_guess=1.0, lower_bound=0.0)
symbolic = [margin_below("x", x, 2.0)]
try:
    margin_report(symbolic)
except Exception as error:
    print("\nsymbolic without value_of ->", type(error).__name__)
opti.subject_to([m.value >= 0 for m in symbolic]); opti.minimize(-x)
sol = opti.solve(verbose=False)
print("after a solve:", margin_report(symbolic, sol.value))

-0.0667  compatible=False  battery current_A
+0.0000  compatible=True  static_margin
+0.2000  compatible=True  motor torque_Nm

symbolic without value_of -> RuntimeError
after a solve: (MarginReportEntry(label='x', value=-9.994999938811588e-09, compatible=False),)


---
## Gotchas and review questions

- **Counts are Python ints, never variables.** Number of lanes, buses, strings, rotors: all structure. Changing them
  means rebuilding the problem (an enumeration), not optimizing.
- **`Bus.count` is invisible to the residuals.** Two buses are lumped as one junction; only redundancy logic reads it.
- **Direction is a sign convention.** Negative flow on an OUT port is legal (battery charging).
- **The topology never calls its components.** `Dummy()` objects above were accepted; the component type matters only
  to `powertrain/ports.py` (which ports it gets) and to the physics code that evaluates it.
- **`compatible` has no tolerance.** After a solve, a binding margin is typically −1e-8 (IPOPT's tolerance), so the
  report flags it `compatible=False` (see the last line above). The sizing reads binding constraints with
  |margin| < 1e-4 and treats −1e-6 as satisfied; read the flag with that in mind.
- **Margins are normalized by the limit**, so their values are comparable across physics and of order one for IPOPT;
  a limit of zero or a negative limit is not guarded.

<details><summary>Why does a port connect only once, but a bus many times?</summary>

A direct connection is a pairwise identity (speed = speed, torque = torque). A junction of three or more ports needs a
conservation law (KCL), which only a bus writes. Forbidding a second direct connection forces junctions to be
declared, so conservation is never forgotten.
</details>

<details><summary>What is the effort/flow distinction and where is it used?</summary>

Efforts (speed, voltage) are equal across a junction; flows (torque, current, fuel flow) add. `port_flow_fields`
names the flow per domain, and `connection_residuals` uses it for combiners and splitters: efforts equal, total flow
conserved.
</details>

<details><summary>Why are margins not clipped or used as penalties?</summary>

A clip (`min(value, limit)`) is non-smooth and hides the active limit. A penalty changes the objective. A plain
inequality `margin ≥ 0` keeps the problem smooth, lets IPOPT report the multiplier, and lets the result name exactly
which limit is binding.
</details>

In [12]:
check_summary()

1 of 1 checks passed
